In [1]:
# Cell 1: Read the silver_sales Delta table
df_silver = spark.read.format('delta').load('Tables/dbo/silver_sales')

print(f'Row count: {df_silver.count()}')
print(f'Columns: {df_silver.columns}')
display(df_silver.limit(5))

StatementMeta(, 1e4e5315-4152-4450-b1bd-e44643c13365, 3, Finished, Available, Finished, False)

Row count: 1268
Columns: ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'revenue', 'quantity', 'status', 'revenue_usd']


SynapseWidget(Synapse.DataFrame, 1072e62e-4750-4beb-8883-ca38ef221c8e)

In [3]:
# Cell 2: Extract year and moth from order_date
from pyspark.sql.functions import col, date_format, sum, count, round, avg

# Add year_month column for grouping 
df_dated = df_silver.withColumn(
    'year_month',
    date_format(col('order_date'), 'yyyy-MM')
)

display(df_dated.select('order_id', 'order_date', 'year_month').limit(5))

StatementMeta(, 1e4e5315-4152-4450-b1bd-e44643c13365, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e59252cc-ccd4-4363-8824-28a19f6b47e1)

In [4]:
# Cell 3: Aggregate to Gold layer
df_gold = df_dated.groupBy('region', 'year_month') \
    .agg(
        round(sum('revenue'), 2).alias('total_revenue'),
        count('order_id').alias('total_orders'),
        round(avg('revenue'), 2).alias('avg_order_value')
    ) \
    .orderBy('year_month', 'region')

print(f'Gold table row count: {df_gold.count()}')
display(df_gold)

StatementMeta(, 1e4e5315-4152-4450-b1bd-e44643c13365, 6, Finished, Available, Finished, False)

Gold table row count: 96


SynapseWidget(Synapse.DataFrame, 3151bb07-a83a-4f24-a228-d5c0f17790a3)

In [5]:
# Cell 4: Write the Gold layer to the Delta table
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gold_sales_summary')

StatementMeta(, 1e4e5315-4152-4450-b1bd-e44643c13365, 7, Finished, Available, Finished, False)